Food waste management

In [3]:
import pandas as pd
import numpy as np

In [4]:
#Importing the datasets

Providers_data= pd.read_csv('providers_data.csv')
Receivers_data=pd.read_csv('receivers_data.csv')
Foodlisting_data=pd.read_csv('food_listings_data.csv')
Claims_data=pd.read_csv('claims_data.csv')

In [5]:
Providers_data

,Provider_ID,Name,Type,Address,City,Contact
0,1,Gonzales-Cochran,Supermarket,"74347 Christopher Extensions\nAndreamouth, OK ...",New Jessica,+1-600-220-0480
1,2,"Nielsen, Johnson and Fuller",Grocery Store,"91228 Hanson Stream\nWelchtown, OR 27136",East Sheena,+1-925-283-8901x6297
2,3,Miller-Black,Supermarket,"561 Martinez Point Suite 507\nGuzmanchester, W...",Lake Jesusview,001-517-295-2206
3,4,"Clark, Prince and Williams",Grocery Store,"467 Bell Trail Suite 409\nPort Jesus, IA 61188",Mendezmouth,556.944.8935x401
4,5,Coleman-Farley,Grocery Store,"078 Matthew Creek Apt. 319\nSaraborough, MA 53978",Valentineside,193.714.6577
...,...,...,...,...,...,...
995,996,"Vasquez, Ruiz and Flowers",Restaurant,"84308 Justin Stravenue\nNew Amberside, NE 53447",Williamview,+1-319-378-7627x0682
996,997,Garza-Williams,Catering Service,"08864 Figueroa Radial Suite 948\nJennaberg, AZ...",East Rossside,001-924-441-3963x746
997,998,Novak Group,Grocery Store,"934 Zachary Run\nMelissamouth, WY 02729",Joshuastad,(903)642-1969x3300
998,999,Moody Ltd,Grocery Store,"17580 Ernest Hills\nLake Michaelmouth, OR 56416",Stevenchester,637.300.3664x4880


In [ ]:
Receivers_data

In [ ]:
Foodlisting_data

In [ ]:
Claims_data

In [7]:
Providers_data.isnull().sum()

Provider_ID    0
Name           0
Type           0
Address        0
City           0
Contact        0
dtype: int64

In [8]:
Receivers_data.isnull().sum()

Receiver_ID    0
Name           0
Type           0
City           0
Contact        0
dtype: int64

In [9]:
Foodlisting_data.isnull().sum()

Food_ID          0
Food_Name        0
Quantity         0
Expiry_Date      0
Provider_ID      0
Provider_Type    0
Location         0
Food_Type        0
Meal_Type        0
dtype: int64

In [10]:
Claims_data.isnull().sum()

Claim_ID       0
Food_ID        0
Receiver_ID    0
Status         0
Timestamp      0
dtype: int64

In [11]:
from mysql import connector
import streamlit as st

In [12]:
 connection= connector.connect(
            host="localhost",
            user="root",
            password="12345678"
        )
mycursor=connection.cursor()
mycursor

query="drop database food_waste_management"
mycursor.execute(query)

In [13]:
query="Create database IF NOT EXISTS food_waste_management"
mycursor.execute(query)

In [14]:
query="show databases"
mycursor.execute(query)
for db in mycursor:
    print(db)

('customer_sales',)
('food_waste_management',)
('information_schema',)
('mini_pro_2',)
('mysql',)
('performance_schema',)
('sys',)


In [15]:
query= "use food_waste_management"
mycursor.execute(query)

In [16]:
mycursor.execute("""
    CREATE TABLE IF NOT EXISTS providers(
        Provider_ID INT PRIMARY KEY,
        Name VARCHAR(100),
        Type VARCHAR(255),
        Address TEXT,
        City VARCHAR(100),
        Contact VARCHAR(50)
    )   
""")
connection.commit()

In [ ]:
#insert data using iterrows()
for index,row in Providers_data.iterrows():
    mycursor.execute('''
        insert into providers(Provider_ID,Name,Type,Address,City,Contact)
        values(%s,%s,%s,%s,%s,%s)
    ''' ,tuple(row))

In [ ]:
query="SELECT * FROM food_waste_management.providers"
mycursor.execute(query)
for data in mycursor:
    print(data)

query="drop table providers"
mycursor.execute(query)

In [116]:
mycursor.execute('''
    CREATE TABLE IF NOT EXISTS receivers(
        receivers_ID INT PRIMARY KEY,
        Name VARCHAR(100),
        Type VARCHAR(255),
        City VARCHAR(100),
        Contact VARCHAR(50)
    )
    
''')
connection.commit()

In [ ]:
#insert data using iterrows()
for index,row in Receivers_data.iterrows():
    mycursor.execute("""
        insert into receivers(receivers_ID,Name,Type,City,Contact)
        values(%s,%s,%s,%s,%s)
    """ ,tuple(row))
connection.commit()

In [ ]:
query="SELECT * FROM food_waste_management.receivers"
mycursor.execute(query)
for data in mycursor:
    print(data)

In [ ]:

# Convert the date column to date format
Foodlisting_data["Expiry_Date"] = pd.to_datetime(Foodlisting_data["Expiry_Date"])

In [ ]:

import datetime
from dateutil.parser import parse
from datetime import datetime

query="drop table foodlisting"
mycursor.execute(query)

In [ ]:
#Foodlisting_data
mycursor.execute('''
    CREATE TABLE IF NOT EXISTS foodlisting(
        Food_ID INT PRIMARY KEY,
        Food_Name VARCHAR(50),
        Quantity INT,
        Expiry_Date DATE,
        Provider_ID INT,
        Provider_Type VARCHAR(100),
        Location VARCHAR(100),
        Food_Type VARCHAR(50),
        Meal_Type VARCHAR(50)
        
    )
    
''')
connection.commit()

In [ ]:
#insert data using iterrows()

Foodlisting_data["Expiry_Date"]  = Foodlisting_data["Expiry_Date"].dt.strftime("%Y-%m-%d")

for index,row in Foodlisting_data.iterrows():
    mycursor.execute("""
        insert into foodlisting (Food_ID,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type)
        values(%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """ ,tuple(row))

In [ ]:
query="SELECT * FROM food_waste_management.foodlisting"
mycursor.execute(query)
for data in mycursor:
    print(data)

In [ ]:
# Convert the date column to date format
Claims_data["Timestamp"] = pd.to_datetime(Claims_data["Timestamp"])
from datetime import datetime
Claims_data["Timestamp"] = Claims_data["Timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")  # Convert to string

In [ ]:
mycursor.execute("""
    CREATE TABLE IF NOT EXISTS claims(
        Claim_ID INT PRIMARY KEY,
        Food_ID INT,
        Receiver_ID INT,
        Status TEXT,
        Timestamp TIMESTAMP
    )   
""")
connection.commit()

In [ ]:
#insert data using iterrows()
for index,row in Claims_data.iterrows():
    mycursor.execute("""
        insert into claims(Claim_ID,Food_ID,Receiver_ID,Status,Timestamp)
        values(%s,%s,%s,%s,%s)
    """ ,tuple(row))

In [ ]:
query="SELECT * FROM food_waste_management.claims"
mycursor.execute(query)
for data in mycursor:
    print(data)

Streamlit

In [ ]:
%%writefile foodwasted.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


#st.markdown(":blue-badge[Home]")
st.badge("Home", color="blue")

st.markdown(
    
  '<style>div.block-container {background-color: #E6E6FA;fullpage-color=#E6E6FA;} </style> ',
    unsafe_allow_html=True
)
#SQL Connection
import mysql.connector
#def get_data(query, params=None):
mydb = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "12345678",
    database='food_waste_management',
    autocommit = True)
mycursor= mydb.cursor()
   
#here is the sidebar
menu = st.sidebar.radio('Navigation',['Home','Food Provider','FoodReceiver','Food listing','Claims','sql query'])
selection = st.sidebar.radio("Go to", menu)
#-------------------home page--------------
if selection == 'Home':
    st.title('FOOD WASTE MANAGEMENT')
    st.image("food.jpg")
    st.subheader("Welcome to the Food Donation Management System")
    st.write("""
    This system helps manage food donations by displaying available food listings, tracking food claims, 
    and connecting food providers and receivers.
    """)
    st.write("Here displaying how much food available from all the providers")
    query1 = "SELECT f.Location,SUM(f.Quantity) AS Total_Quantity,COUNT(DISTINCT f.Provider_ID) AS Total_Providers,COUNT(DISTINCT r.receivers_ID) AS Total_Receivers FROM foodlisting f JOIN receivers r ON f.Location = r.City GROUP BY f.Location ORDER BY f.Location DESC Limit 10;"

    mycursor.execute(query1)
    result1 = mycursor.fetchall()

# Convert to DataFrame
    df_city_summary = pd.DataFrame(result1, columns=["City", "Total_Quantity", "Total_Providers", "Total_Receivers"])

# Display in Streamlit
    st.write("###Total Food Providers and Receivers in a City and total quantity of food")
    st.dataframe(df_city_summary)

    df_melted = df_city_summary.melt(id_vars=["City"], 
                                 value_vars=["Total_Quantity", "Total_Providers", "Total_Receivers"],
                                 var_name="Metric", value_name="Value")

    st.write("### Distribution of Metrics by City")
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_melted, x="Metric", y="Value", palette="pastel")
    st.pyplot(plt.gcf())
    st.write("From the graph we can see how much food available.SO Use this Application to claim your food today")

#----------Provider---------------
elif selection == 'Food Provider':
    st.subheader("Food Providers")
    
    # Filter Providers by Type
    provider_type = st.selectbox("Filter by Provider Type", ['All', 'Restaurant', 'Grocery Store','Supermarket'])
    if  provider_type=='Restaurant':
        query="SELECT * FROM food_waste_management.providers WHERE Type='Restaurant';"
        mycursor.execute(query)
        data = mycursor.fetchall()
        providers=pd.DataFrame(data)
        st.write(providers)
        
    elif provider_type=='Grocery Store':
        query="select * from food_waste_management.providers p where p.Type='Grocery Store';"
        mycursor.execute(query)
         # Fetch data and store it in a Pandas DataFrame
        data = mycursor.fetchall()
        providers=pd.DataFrame(data)
        st.write(providers)
    elif provider_type=='Supermarket':
        query="select * from food_waste_management.providers p where p.Type='Supermarket';"
        mycursor.execute(query)
        data = mycursor.fetchall()
        providers=pd.DataFrame(data)
        st.write(providers)
    else:
        query="select * from food_waste_management.providers;"
        mycursor.execute(query)
         # Fetch data and store it in a Pandas DataFrame
        data = mycursor.fetchall()
        providers=pd.DataFrame(data)
        st.write(providers)
 # ---------------Allow CRUD Operations (Add/Update/Remove Provider)-----------------
    
    menu = st.selectbox("crud operation for providers", ["Add Donation", "Update Donation", "Delete Donation"])

    if menu == "Add Donation":
        st.subheader("Add a New Food Donation")
        ID=st.text_input('provider_ID')
        provider = st.text_input("Provider Name")
        Food_Type = st.text_input("Food Type")
        location = st.text_input("Location")
        city= st.text_input("City")
        contact = st.text_input("Contact Info")
        if st.button("Submit"):
            query="insert into providers(Provider_ID,Name,Type,address,city,contact) values (%s,%s,%s,%s,%s,%s)"
            values=(ID,provider,Food_Type,location,city,contact)
            mycursor.execute(query,values)
            st.success("Donation added successfully!")
        #  Fetch and display updated record
        fetch_query = "SELECT * FROM providers WHERE Provider_ID = %s AND Name = %s"
        mycursor.execute(fetch_query, (ID, provider))
        result = mycursor.fetchone()

        if result:
            st.write("### Updated Donation Details:")
            st.write(f"Provider ID: {result[0]}")
            st.write(f"Name: {result[1]}")
            st.write(f"Food Type: {result[2]}")
            st.write(f"Address: {result[3]}")
            st.write(f"City: {result[4]}")
            st.write(f"Contact: {result[5]}")
        else:
            st.warning("No matching record found.")
            
    elif menu =="Update Donation":
        st.subheader('Update a food donation')
        
        ID=st.text_input('provider_ID')
        provider = st.text_input("Provider Name")
        new_Food_Type=st.text_input(" Food Type")
       
        if st.button("Submit"):
            query = "UPDATE providers SET Type= %s WHERE Provider_ID = %s AND Name = %s"

            mycursor.execute(query,(new_Food_Type,ID,provider))
           # connection.commit()
            #insert_data(provider, location, food_type, quantity, contact)
            st.success("Donation updated successfully!")
             #  Fetch and display updated record
        fetch_query = "SELECT * FROM providers WHERE Provider_ID = %s AND Name = %s"
        mycursor.execute(fetch_query, (ID, provider))
        result = mycursor.fetchone()

        if result:
            st.write("### Updated Donation Details:")
            st.write(f"Provider ID: {result[0]}")
            st.write(f"Name: {result[1]}")
          
           
        else:
            st.warning("No matching record found.")
            
    elif menu == "Delete Donation":
        st.subheader('Delete a food donation')
        ID=st.text_input('provider_ID')
        provider = st.text_input("Provider Name")
        if st.button("Submit"):
            query = "DELETE FROM providers WHERE Provider_ID = %s AND Name = %s"
        
        # Execute the query with the actual values
            mycursor.execute(query, (ID, provider))          
            st.success("Deletion updated successfully!")

    from tabulate import tabulate
    import seaborn as sns
    query = "SELECT p.City,COUNT(DISTINCT p.Provider_ID) AS food_providers,COUNT(DISTINCT r.receivers_ID) AS food_receivers FROM providers p JOIN receivers r ON p.city = r.city GROUP BY p.City ORDER BY food_providers DESC;"
    mycursor.execute(query)
    result = mycursor.fetchall()
    
    # ✅ Convert to DataFrame with correct column names
    df_providers = pd.DataFrame(result, columns=["City", "food_providers", "food_receivers"])
    
    # ✅ Display DataFrame to check column names
    st.dataframe(df_providers)
    
    # ✅ Line plot for providers
    st.write("### Food Provider Trends by City")
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df_providers, x="City", y="food_providers")
    plt.xticks(rotation=45)
    plt.tight_layout()
    st.pyplot(plt.gcf())


    # -----------------Receivers Page-------------------------
elif selection == 'FoodReceiver':
    st.subheader("Food Receivers")
    
    # Filter Receivers by Type
    receiver_type = st.selectbox("Filter by Receiver Type", ['All', 'NGO', 'Community Center', 'Individual'])
    if  receiver_type=='NGO':
        query="SELECT * FROM food_waste_management.receivers WHERE Type='NGO';"
        mycursor.execute(query)
        data = mycursor.fetchall()
        receivers=pd.DataFrame(data)
        st.write(receivers)
        
    elif receiver_type=='Community Center':
        query="select * from food_waste_management.receivers  where Type='Community Center';"
        mycursor.execute(query)
         # Fetch data and store it in a Pandas DataFrame
        data = mycursor.fetchall()
        receivers=pd.DataFrame(data)
        st.write(receivers)
    elif receiver_type=='Individual':
        query="select * from food_waste_management.receivers where Type='Individual';"
        mycursor.execute(query)
         # Fetch data and store it in a Pandas DataFrame
        data = mycursor.fetchall()
        receivers=pd.DataFrame(data)
        st.write(receivers)
    else:
        query="select * from food_waste_management.receivers;"
        mycursor.execute(query)
         # Fetch data and store it in a Pandas DataFrame
        data = mycursor.fetchall()
        receivers=pd.DataFrame(data)
        st.write(receivers)

    st.subheader("Update Receiver Form")

        # Use st.form to group inputs and submission
    with st.form("update_receiver_form"):
        receiver_id = st.number_input("Receiver ID to Update", min_value=1)
        name = st.text_input("New Name")
        receiver_type = st.selectbox("Receiver Type",["   ","NGO","Individual","shelter","chariry"])
        contact = st.text_input("New Contact Info")
        city = st.text_input("New City")
        
        submitted = st.form_submit_button("Submit")
        
        # Run database query after form is submitted
    if submitted:
        query = "UPDATE receivers SET Name = %s, Type = %s, Contact = %s, City = %s WHERE receivers_ID = %s"
        values = (name, receiver_type, contact, city, receiver_id)
        mycursor.execute(query, values)
        st.success("Receiver updated successfully!")
        
            # Fetch and display updated record
        fetch_query = "SELECT * FROM receivers WHERE receivers_ID = %s"
        mycursor.execute(fetch_query, (receiver_id,))
        result = mycursor.fetchone()
        if result:
            st.write("Updated Record:", result)
        else:
            st.warning("No matching record found.")

# -------------------Food Listings Page------------------------------
elif selection == 'Food listing':
    st.subheader("Food Listings")

    # Filter by food type or meal type
    food_type = st.selectbox("Filter by Food Type", ['All', 'Vegetarian', 'Non-Vegetarian', 'Vegan'])
    if food_type == 'Vegetarian':
        query="select * from foodlisting where Food_Type = 'Vegetarian';"
        mycursor.execute(query)
        st.write(mycursor)
    elif food_type == 'Non-Vegetarian':
        query="select * from foodlisting where Food_Type = 'Non-Vegetarian';"
        mycursor.execute(query)
        st.write(mycursor)
    elif food_type == 'Vegan':
        query="select * from foodlisting where Food_Type = 'Vegan';"
        mycursor.execute(query)
        st.write(mycursor)
    else:
        query="select * from foodlisting;"
        mycursor.execute(query)
        st.write(mycursor)
    
    meal_type = st.selectbox("Filter by Meal Type", ['All', 'Breakfast', 'Lunch', 'Dinner', 'Snacks'])
    if meal_type == 'Breakfast':
        query="select * from foodlisting where Meal_Type = 'Breakfast';"
        mycursor.execute(query)
        st.write(mycursor)
    elif meal_type == 'Lunch':
        query="select * from foodlisting where Meal_Type = 'Lunch';"
        mycursor.execute(query)
        st.write(mycursor)
    elif meal_type == 'Dinner':
        query="select * from foodlisting where Meal_Type = 'Dinner';"
        mycursor.execute(query)
        st.write(mycursor)
    elif meal_type == 'Snacks':
        query="select * from foodlisting where Meal_Type = 'Snacks';"
        mycursor.execute(query)
        st.write(mycursor)
        
    else:
        query="select * from foodlisting;"
        mycursor.execute(query)
        st.write(mycursor)

#============== add  foodlisting ======================================
    menu = st.selectbox("Update Foodlisting", ["Add Food Listing", "delete foodlisting"])
    if menu == "Add Food Listing":
        with st.form("food_listing_form"):
            food_name = st.text_input("Food Name")
            quantity = st.number_input("Quantity", min_value=1)
            expiry_date = st.date_input("Expiry Date")
            provider_id = st.text_input("Provider ID")
    
            provider_type = st.selectbox("Provider Type", [
                "Catering Service", 
                "Grocery Store", 
                "Restaurant", 
                "Supermarket"
            ])
    
            location = st.text_input("Location")
    
            food_type = st.selectbox("Food Type", [
                "Non-Vegetarian",
                "Vegan",
                "Vegetarian"
            ])
    
            meal_type = st.selectbox("Meal Type", [
                "Breakfast",
                "Lunch",
                "Dinner"
            ])
    
            submitted = st.form_submit_button("Submit")
    
            if submitted:
                for index,row in Foodlisting_data.iterrows():
                    mycursor.execute(
                        """
                        INSERT INTO foodlisting (
                            Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type
                        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                        """,tuple(row)
                        (
                            food_name, quantity, expiry_date,
                            provider_id, provider_type, location,
                            food_type, meal_type
                        )
                    )
                st.success("✅ Food listing added successfully!")

                #  Fetch and display updated record
        fetch_query = """SELECT * FROM foodlisting WHERE Food_Name=%s AND Quantity=%s AND Expiry_Date=%s AND Provider_ID = %s AND Provider_ID=%s AND Location=%s AND Food_Type=%s AND Meal_Type=%s"""
        mycursor.execute(fetch_query, (food_name, quantity, expiry_date,
                            provider_id, provider_type, location,
                            food_type, meal_type))
        result = mycursor.fetchone()

# === DELETE FOOD LISTING ===
    elif menu == "Delete Food Listing":
        try:
            df = fetch_data("Food_listing")
            if df.empty:
                st.warning("⚠️ No food listings found to delete.")
            else:
                # Show the table to the user
                st.dataframe(df)
    
                # Let user pick an item to delete by Food_ID
                food_ids = df["Food_ID"].tolist()  # Adjust column name if needed
                selected_id = st.selectbox("Select Food ID to delete", food_ids)
    
                if st.button("Delete"):
                    mycursor.execute("DELETE FROM food WHERE Food_ID = ?", (selected_id,))
                    connection.commit()
                    st.success(f"✅ Deleted Food Listing with ID {selected_id}")
        except Exception as e:
            st.error(f"❌ Error fetching or deleting data: {e}")


       


#----------------- Claims Page--------------------
elif selection == 'Claims':
    st.subheader("Food Claims")
    
    # Filter by claim status
    claim_status = st.selectbox("Filter by Claim Status", ['All', 'Pending', 'Completed', 'Cancelled'])
    if claim_status == 'Pending':
        query="select * from claims where Status='Pending';"
        mycursor.execute(query)
        st.write(mycursor)
    elif claim_status == 'Completed':
        query="select * from claims where Status='Completed';"
        mycursor.execute(query)
        st.write(mycursor)
    elif claim_status == 'Cancelled':
        query="select * from claims where Status='Cancelled';"
        mycursor.execute(query)
        st.write(mycursor)
    else:
        query="select * from claims;"
        mycursor.execute(query)
        st.write(mycursor)

    with st.form("update_claim_form"):
        claim_id = st.number_input("Claim ID to Update")
        status = st.selectbox("New Status", ["Pending", "Completed", "Canceled"])
        submitted = st.form_submit_button("Submit")
        if submitted:
            mycursor.execute(
                """
                UPDATE claims 
                SET Status = ?, Timestamp = DATETIME('now') 
                WHERE Claim_ID = ?
                """,
                (status, claim_id)
            )
            st.success("✅ Claim updated successfully with current timestamp!")


#--------------------SQL queries----------------------------------------------
elif selection == 'sql query':
    st.title("📋 SQL Query Results")
    
    queries = {
        "1. How many food providers and receivers are there in each city?":
            "SELECT p.City, COUNT(DISTINCT p.Provider_ID) AS food_providers, COUNT(DISTINCT r.receivers_ID) AS food_receivers FROM providers p JOIN receivers r ON p.city = r.city GROUP BY p.City ORDER BY food_providers DESC;",

        "2. Which type of food provider contributes the most food?":
            "SELECT fp.Provider_Type, SUM(fp.Quantity) AS total_food_contributed FROM foodlisting fp GROUP BY fp.Provider_Type ORDER BY total_food_contributed DESC;",

        "3. What is the contact information of food providers in a specific city?":
            "SELECT City, Contact FROM providers ORDER BY City;",
        "4.Which receivers have claimed the most food?":
           " SELECT r.Name AS receiver_name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN receivers r ON c.Receiver_ID = r.receivers_ID WHERE c.Status = 'Completed' GROUP BY r.Name ORDER BY total_claims DESC LIMIT 1;",
        "5.What is the total quantity of food available from all providers?":"select sum(Quantity),count(Provider_ID) from foodlisting;",
        "6.Which city has the highest number of food listings?":"select Location,Count(Provider_ID) as foodlisting from foodlisting Group by Location Order By foodlisting desc limit 2;",
        "7.What are the most commonly available food types?":"select Food_Name ,count(Food_Name) as mostavailable from foodlisting  group by Food_Name order by mostavailable desc limit 5;",
        "8.Which food listings are expiring soon (within the next 3 days)?":"select Food_name,Expiry_Date from foodlisting f WHERE Expiry_Date BETWEEN 2025-03-17 AND DATE_ADD(2025-03-17, INTERVAL 5 DAY);",
        "9.How many food claims have been made for each food item?":"select f.Food_Name,count(c.Claim_ID) as claim from claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID group by f.Food_Name Order by claim desc;",
        "10.Which provider has had the highest number of successful food claims?":"SELECT p.Name AS provider_name, COUNT(c.Claim_ID) AS successful_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID JOIN providers p ON f.Provider_ID = p.Provider_ID WHERE c.Status = 'Completed' GROUP BY p.Name ORDER BY successful_claims DESC LIMIT 1;",
        "11.Which city has the fastest claim rate (measured by average time between food listing and claim)?":"SELECT f.Location AS city, AVG(c.Timestamp - f.Expiry_Date) AS avg_claim_time FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Location ORDER BY avg_claim_time desc  -- Fastest claim time first LIMIT 1;",
        "12.What percentage of food claims are completed vs. pending vs. canceled?":"SELECT  c.Status,COUNT(c.Claim_ID) AS total_claims, ROUND((COUNT(c.Claim_ID) * 100.0) / (SELECT COUNT(*) FROM claims), 2) AS percentage FROM claims c GROUP BY c.Status;",
        "13.What is the average quantity of food claimed per receiver?":"SELECT AVG(food_claimed.total_quantity) AS avg_quantity_per_receiver FROM ( SELECT c.Receiver_ID, SUM(f.Quantity) AS total_quantity FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY c.Receiver_ID Order BY c.Receiver_ID) AS food_claimed;",
        "14.Which meal type (breakfast, lunch, dinner, snacks) is claimed the most?":"SELECT f.Meal_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' claims GROUP BY f.Meal_Type ORDER BY total_claims DESC LIMIT 1;",
        "15.What is the total quantity of food donated by each provider?":"SELECT p.Name AS provider_name, SUM(f.Quantity) AS total_quantity_donated FROM foodlisting f JOIN providers p ON f.Provider_ID = p.Provider_ID GROUP BY p.Name ORDER BY total_quantity_donated DESC;",
        "16. Which food type is claimed the least?":"SELECT f.Food_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Type ORDER BY total_claims ASC LIMIT 1;",
        "17.How many claims have been made for food expiring in the next 7 days?":"SELECT COUNT(c.Claim_ID) AS claims_for_expiring_food FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE f.Expiry_Date BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL 7 DAY) AND c.Status = 'Completed';",
       " 18. How many claims have been made for each food item?":"SELECT f.Food_Name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Name ORDER BY total_claims DESC;",
       "19 Which city has the most active receivers?":"SELECT r.City, COUNT(DISTINCT r.receivers_ID) AS total_receivers FROM receivers r JOIN claims c ON r.receivers_ID = c.Receiver_ID WHERE c.Status = 'Completed' GROUP BY r.City ORDER BY total_receivers DESC LIMIT 1;",
       "20 Which receiver has claimed the most food?":"SELECT r.Name AS receiver_name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN receivers r ON c.Receiver_ID = r.receivers_ID WHERE c.Status = 'Completed' GROUP BY r.Name ORDER BY total_claims DESC LIMIT 1;",
       "21.Which provider has donated the highest quantity of a specific food type?":"SELECT p.Name AS provider_name, f.Food_Type, SUM(f.Quantity) AS Food_Type FROM foodlisting f JOIN providers p ON f.Provider_ID = p.Provider_ID GROUP BY p.Name, f.Food_Type ORDER BY total_donated DESC LIMIT 5;",
       "22. Which food type (e.g., vegetarian, vegan,non-veg) is most commonly claimed?":"SELECT f.Food_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Type ORDER BY total_claims DESC LIMIT 1;",
        
    }

    selected_query = st.selectbox("Choose a Query", list(queries.keys()))
    try:
        query_result = pd.read_sql(queries[selected_query], mydb)
        st.write("### Query Result:")
        st.dataframe(query_result)
    except Exception as e:
        st.error(f"An error occurred: {e}")
    finally:
        mydb.close()
    
 













        

Overwriting foodwasted.py


In [ ]:
!streamlit run foodwasted.py

^C
  File "/Users/ramreddy/Library/Python/3.9/lib/python/site-packages/streamlit/runtime/memory_session_storage.py", line 21, in <module>
    from streamlit.runtime.session_manager import SessionInfo, SessionStorage
  File "<frozen importlib._bootstrap>", line 1007, in _find_and_load
  File "<frozen importlib._bootstrap>", line 986, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 680, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 846, in exec_module
  File "<frozen importlib._bootstrap_external>", line 978, in get_code
  File "<frozen importlib._bootstrap_external>", line 647, in _compile_bytecode
KeyboardInterrupt


In [ ]:
#1.How many food providers and receivers are there in each city?
from tabulate import tabulate
query = "SELECT p.City,COUNT(DISTINCT p.Provider_ID) AS food_providers,COUNT(DISTINCT r.receivers_ID) AS food_receivers FROM providers p JOIN receivers r ON p.city = r.city GROUP BY p.City ORDER BY food_providers DESC;"
mycursor.execute(query)
result = mycursor.fetchall()

# Define table headers
headers = ["City", "Providers","Receivers"]

# Print the result as a table
print(tabulate(result, headers=headers, tablefmt="grid"))


+--------------------+-------------+-------------+
| City               |   Providers |   Receivers |
+====================+=============+=============+
| West Christopher   |           2 |           1 |
+--------------------+-------------+-------------+
| Port Melissa       |           2 |           1 |
+--------------------+-------------+-------------+
| North Michelle     |           2 |           1 |
+--------------------+-------------+-------------+
| New Daniel         |           2 |           1 |
+--------------------+-------------+-------------+
| Lake Michael       |           2 |           1 |
+--------------------+-------------+-------------+
| Davidport          |           1 |           1 |
+--------------------+-------------+-------------+
| East Ashleyshire   |           1 |           1 |
+--------------------+-------------+-------------+
| East Emily         |           1 |           1 |
+--------------------+-------------+-------------+
| East John          |         

2.Which type of food provider (restaurant, grocery store, etc.) contributes the most food?

In [ ]:
from tabulate import tabulate

query1 = "SELECT fp.Provider_Type, SUM(fp.Quantity) AS total_food_contributed FROM foodlisting fp GROUP BY fp.provider_type ORDER BY total_food_contributed DESC;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Provider TYpe", "Quantity"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))


+------------------+------------+
| Provider TYpe    |   Quantity |
+==================+============+
| Restaurant       |       6923 |
+------------------+------------+
| Supermarket      |       6696 |
+------------------+------------+
| Catering Service |       6116 |
+------------------+------------+
| Grocery Store    |       6059 |
+------------------+------------+


#3.What is the contact information of food providers in a specific city?

In [ ]:
from tabulate import tabulate

query1 = "select City,p.Contact as contact from providers p order by city;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["City", "Contact"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))


+--------------------------+------------------------+
| City                     | Contact                |
+==========================+========================+
| Adambury                 | 6703380260             |
+--------------------------+------------------------+
| Adamsview                | 001-281-026-8022       |
+--------------------------+------------------------+
| Adamsville               | (112)122-3591x558      |
+--------------------------+------------------------+
| Aguirreville             | 8228891240             |
+--------------------------+------------------------+
| Alexanderchester         | 001-867-928-0212x3211  |
+--------------------------+------------------------+
| Alexanderstad            | 084-323-1485           |
+--------------------------+------------------------+
| Allenborough             | 001-590-644-2836       |
+--------------------------+------------------------+
| Allenton                 | 795.078.7850           |
+--------------------------+

In [ ]:
#4.Which receivers have claimed the most food?

In [ ]:
query1 = "SELECT r.Name AS receiver_name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN receivers r ON c.Receiver_ID = r.receivers_ID WHERE c.Status = 'Completed' GROUP BY r.Name ORDER BY total_claims DESC LIMIT 1;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Receiver_name", "Total_claims"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+-----------------+----------------+
| Receiver_name   |   Total_claims |
+=================+================+
| Derek Potter    |              3 |
+-----------------+----------------+


In [ ]:
#5.What is the total quantity of food available from all providers?

In [ ]:
query1 = "select sum(Quantity) as Total_Quantity,count(Provider_ID) as Total_Providers FROM foodlisting;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Total_Quantity", "Total_Providers"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+------------------+-------------------+
|   Total_Quantity |   Total_Providers |
+==================+===================+
|            25794 |              1000 |
+------------------+-------------------+


In [ ]:
#6.Which city has the highest number of food listings?

In [ ]:


query1 = "select Location,Count(Provider_ID) as foodlisting from foodlisting Group by Location Order By foodlisting desc limit 2;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Location","FoodListing"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+---------------+---------------+
| Location      |   FoodListing |
+===============+===============+
| South Kathryn |             6 |
+---------------+---------------+
| New Carol     |             6 |
+---------------+---------------+


In [ ]:
#7.What are the most commonly available food types?

In [ ]:
query1 = "select Food_Name ,count(Food_Name) as mostavailable from foodlisting  group by Food_Name order by mostavailable desc limit 5;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Food_name","count"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

In [ ]:
#8.Which food listings are expiring soon (within the next 3 days)?

In [ ]:


query1 = "select Food_name,Expiry_Date from foodlisting f WHERE Expiry_Date BETWEEN 2025-03-17 AND DATE_ADD(2025-03-17, INTERVAL 5 DAY);"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Food_name","exprity_date"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))








+-------------+----------------+
| Food_name   | exprity_date   |
+=============+================+
+-------------+----------------+


select * from foodlisting f where DATEDIFF(day, GETDATE(),f.Expiry_Date) < 3;

select Food_Name  from foodlisting where Expiry_Date > DATEADD(day,3,getdate());

select Food_name,Expiry_Date from foodlisting f where DATEDIFF('2025-03-17','2025-03-21') ;"

"select Food_name,Expiry_Date from foodlisting f WHERE Expiry_Date BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL 3 DAY);

SELECT Food_ID, Food_Name, Quantity, Expiry_Date, Provider_ID, Location FROM foodlisting WHERE Expiry_Date BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL 3 DAY) ORDER BY Expiry_Date ASC;

In [ ]:
#9.How many food claims have been made for each food item?

In [ ]:
query1 = "select f.Food_Name,count(c.Claim_ID) as claim from claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID group by f.Food_Name Order by claim desc;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Food_name", "Total_claims"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+-------------+----------------+
| Food_name   |   Total_claims |
+=============+================+
| Rice        |            122 |
+-------------+----------------+
| Soup        |            114 |
+-------------+----------------+
| Dairy       |            110 |
+-------------+----------------+
| Fish        |            108 |
+-------------+----------------+
| Salad       |            106 |
+-------------+----------------+
| Chicken     |            102 |
+-------------+----------------+
| Bread       |             94 |
+-------------+----------------+
| Pasta       |             87 |
+-------------+----------------+
| Vegetables  |             86 |
+-------------+----------------+
| Fruits      |             71 |
+-------------+----------------+


In [ ]:
#10.Which provider has had the highest number of successful food claims?

In [ ]:
query1 = "SELECT p.Name AS provider_name, COUNT(c.Claim_ID) AS successful_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID JOIN providers p ON f.Provider_ID = p.Provider_ID WHERE c.Status = 'Completed' GROUP BY p.Name ORDER BY successful_claims DESC LIMIT 1;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["provider_Name", "Successful_Food_claims"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+-----------------+--------------------------+
| provider_Name   |   Successful_Food_claims |
+=================+==========================+
| Barry Group     |                        5 |
+-----------------+--------------------------+


In [ ]:
#11.Which city has the fastest claim rate (measured by average time between food listing and claim)?

In [ ]:
query1 = "SELECT f.Location AS city, AVG(c.Timestamp - f.Expiry_Date) AS avg_claim_time FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Location ORDER BY avg_claim_time desc  -- Fastest claim time first LIMIT 1; "
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["City", "AVG_Claimtime"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+--------------------------+-----------------+
| City                     |   AVG_Claimtime |
+==========================+=================+
| West Miaside             |     2.02503e+13 |
+--------------------------+-----------------+
| South Lisaberg           |     2.02503e+13 |
+--------------------------+-----------------+
| Brownchester             |     2.02503e+13 |
+--------------------------+-----------------+
| Lake Christopherburgh    |     2.02503e+13 |
+--------------------------+-----------------+
| East Bernard             |     2.02503e+13 |
+--------------------------+-----------------+
| Andersonmouth            |     2.02503e+13 |
+--------------------------+-----------------+
| Brownberg                |     2.02503e+13 |
+--------------------------+-----------------+
| Toddberg                 |     2.02503e+13 |
+--------------------------+-----------------+
| West Lucasville          |     2.02503e+13 |
+--------------------------+-----------------+
| Huberstad  

In [ ]:
#12.What percentage of food claims are completed vs. pending vs. canceled?

In [ ]:
query1 = "SELECT  c.Status,COUNT(c.Claim_ID) AS total_claims, ROUND((COUNT(c.Claim_ID) * 100.0) / (SELECT COUNT(*) FROM claims), 2) AS percentage FROM claims c GROUP BY c.Status; "
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Status", "Total_claims","percentage"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))

+-----------+----------------+--------------+
| Status    |   Total_claims |   percentage |
+===========+================+==============+
| Pending   |            325 |         32.5 |
+-----------+----------------+--------------+
| Cancelled |            336 |         33.6 |
+-----------+----------------+--------------+
| Completed |            339 |         33.9 |
+-----------+----------------+--------------+


In [ ]:
#13.What is the average quantity of food claimed per receiver?

In [ ]:
query1 = "SELECT AVG(food_claimed.total_quantity) AS avg_quantity_per_receiver FROM ( SELECT c.Receiver_ID, SUM(f.Quantity) AS total_quantity FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY c.Receiver_ID Order BY c.Receiver_ID) AS food_claimed;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["avg_quantity_per_receiver","Food claimed"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))



+-----------------------------+
|   avg_quantity_per_receiver |
+=============================+
|                      29.942 |
+-----------------------------+


In [ ]:
#14.Which meal type (breakfast, lunch, dinner, snacks) is claimed the most?

In [ ]:
#Get the meal type with the highest claims
query1 = "SELECT f.Meal_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed'  GROUP BY f.Meal_Type ORDER BY total_claims DESC LIMIT 1;"
mycursor.execute(query1)
result1 = mycursor.fetchall()

# Define table headers
headers = ["Meal_TYpe","exprity_date"]

# Print the result as a table
print(tabulate(result1, headers=headers, tablefmt="grid"))



+-------------+----------------+
| Meal_TYpe   |   exprity_date |
+=============+================+
| Breakfast   |             95 |
+-------------+----------------+


#15.What is the total quantity of food donated by each provider?

In [ ]:
#Sort to see the provider with the highest donation
query="SELECT p.Name AS provider_name, SUM(f.Quantity) AS total_quantity_donated FROM foodlisting f JOIN providers p ON f.Provider_ID = p.Provider_ID GROUP BY p.Name ORDER BY total_quantity_donated DESC; "
mycursor.execute(query)
result=mycursor.fetchall()
headers=["Provider_name","total_quantity_donated"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+-----------------------------------+--------------------------+
| Provider_name                     |   total_quantity_donated |
+===================================+==========================+
| Miller Inc                        |                      217 |
+-----------------------------------+--------------------------+
| Barry Group                       |                      179 |
+-----------------------------------+--------------------------+
| Evans, Wright and Mitchell        |                      158 |
+-----------------------------------+--------------------------+
| Smith Group                       |                      150 |
+-----------------------------------+--------------------------+
| Campbell LLC                      |                      145 |
+-----------------------------------+--------------------------+
| Nelson LLC                        |                      142 |
+-----------------------------------+--------------------------+
| Ruiz-Oneal             

In [ ]:
#16. Which food type is claimed the least?
query="SELECT f.Food_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Type ORDER BY total_claims ASC LIMIT 1;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["Food_Type","total_claims"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+-------------+----------------+
| Food_Type   |   total_claims |
+=============+================+
| Vegan       |             98 |
+-------------+----------------+


In [ ]:
#17. How many claims have been made for food expiring in the next 7 days?
query="SELECT COUNT(c.Claim_ID) AS claims_for_expiring_food FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE f.Expiry_Date BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL 7 DAY) AND c.Status = 'Completed';"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["claims_for_expiring_food"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+----------------------------+
|   claims_for_expiring_food |
+============================+
|                          0 |
+----------------------------+


In [ ]:
#18. How many claims have been made for each food item?
query="SELECT f.Food_Name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Name ORDER BY total_claims DESC;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["Food_Name","total_claims"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+-------------+----------------+
| Food_Name   |   total_claims |
+=============+================+
| Bread       |             42 |
+-------------+----------------+
| Soup        |             41 |
+-------------+----------------+
| Salad       |             37 |
+-------------+----------------+
| Rice        |             36 |
+-------------+----------------+
| Dairy       |             36 |
+-------------+----------------+
| Vegetables  |             33 |
+-------------+----------------+
| Fish        |             33 |
+-------------+----------------+
| Chicken     |             31 |
+-------------+----------------+
| Pasta       |             30 |
+-------------+----------------+
| Fruits      |             20 |
+-------------+----------------+


In [ ]:
#19 Which city has the most active receivers?
query="SELECT r.City, COUNT(DISTINCT r.receivers_ID) AS total_receivers FROM receivers r JOIN claims c ON r.receivers_ID = c.Receiver_ID WHERE c.Status = 'Completed' GROUP BY r.City ORDER BY total_receivers DESC LIMIT 1;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["City","total_receivers"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+--------------+-------------------+
| City         |   total_receivers |
+==============+===================+
| East Michael |                 2 |
+--------------+-------------------+


In [ ]:
#20 Which receiver has claimed the most food?
query="SELECT r.Name AS receiver_name, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN receivers r ON c.Receiver_ID = r.receivers_ID WHERE c.Status = 'Completed' GROUP BY r.Name ORDER BY total_claims DESC LIMIT 1;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["Receiver_name","total_claims"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+-----------------+----------------+
| Receiver_name   |   total_claims |
+=================+================+
| Derek Potter    |              3 |
+-----------------+----------------+


In [ ]:
#21.Which provider has donated the highest quantity of a specific food type?
query="SELECT p.Name AS provider_name, f.Food_Type, SUM(f.Quantity) AS total_donated FROM foodlisting f JOIN providers p ON f.Provider_ID = p.Provider_ID GROUP BY p.Name, f.Food_Type ORDER BY total_donated DESC LIMIT 5;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["provider_name","Food_Type","total_donated"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+---------------------------+----------------+-----------------+
| provider_name             | Food_Type      |   total_donated |
+===========================+================+=================+
| Miller Inc                | Vegan          |             127 |
+---------------------------+----------------+-----------------+
| Hogan-Johnston            | Vegetarian     |              99 |
+---------------------------+----------------+-----------------+
| Wong-Reese                | Non-Vegetarian |              98 |
+---------------------------+----------------+-----------------+
| Bradford-Martinez         | Vegan          |              96 |
+---------------------------+----------------+-----------------+
| Rogers, Harmon and Gordon | Non-Vegetarian |              96 |
+---------------------------+----------------+-----------------+


In [ ]:
#22. Which food type (e.g., vegetarian, vegan,non-veg) is most commonly claimed?
query="SELECT f.Food_Type, COUNT(c.Claim_ID) AS total_claims FROM claims c JOIN foodlisting f ON c.Food_ID = f.Food_ID WHERE c.Status = 'Completed' GROUP BY f.Food_Type ORDER BY total_claims DESC LIMIT 1;"
mycursor.execute(query)
result=mycursor.fetchall()
headers=["provider_name","Food_Type"]
print(tabulate(result, headers=headers, tablefmt="grid"))

+-----------------+-------------+
| provider_name   |   Food_Type |
+=================+=============+
| Vegetarian      |         127 |
+-----------------+-------------+
